In [69]:
# Importing the libraries

from dotenv import load_dotenv 
from anthropic import Anthropic
import json

In [70]:
load_dotenv()

True

In [71]:
client = Anthropic()
model = "claude-haiku-4-5"

In [72]:
# Helper functions

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [73]:
# Dataset Creation Function

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate a comrehensive breakdown of the task at hand into further subtasks.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a 4-5 sub tasks.
* Focus on tasks that are comonly asked to any productivity agent.

Please generate 10 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [74]:
# Calling the generate dataset function

dataset = generate_dataset()
print(dataset)

[{'task': 'Plan and organize a company team building event for 50 people'}, {'task': 'Create a comprehensive marketing strategy for a new product launch'}, {'task': 'Develop an onboarding program for new employees'}, {'task': 'Organize a quarterly business review meeting with stakeholders'}, {'task': 'Plan a cross-functional project to migrate legacy systems to cloud infrastructure'}, {'task': 'Design and implement a customer feedback collection and analysis process'}, {'task': 'Prepare a proposal for securing funding or investment for a business initiative'}, {'task': 'Establish a remote work policy and implementation framework for the organization'}, {'task': 'Create a professional development and training plan for the sales department'}, {'task': 'Develop a disaster recovery and business continuity plan for critical operations'}]


In [75]:
# Saves the generated dataset in the dataset.json file

with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [76]:
# Defining the run prompt function that runs the our prompt we need to evaluate 

def run_prompt(test_case):                 # Test case refers to the test dataset generated above
    """Merges the prompt and the test case input and returns the result"""
    prompt = f"""
    Break the following task into further sub tasks.
    
    {test_case["task"]}
    """
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    add_assistant_message(messages, output)
    return output

In [84]:
# Defining the run_test_case funtion that runs the test cases

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # Grade the output (updated)
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [87]:
# Defining the function that runs the avaluation process and returns the result 

# Updating the run_eval function after implementing the model grader into the pipeline 
# import the mean function from the statistics library 
from statistics import mean 
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
# Add the average score calculator script 
    average_score = mean(result["score"] for result in results)
    print(f"Average Score: {average_score}")
    return results

In [88]:
# To execute our evaluation pipeline, we load our dataset and run it through our functions:

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average Score: 7.25


In [89]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# Team Building Event Planning - Sub-Tasks\n\n## 1. **Define Event Objectives & Scope**\n   - Clarify goals (morale, team bonding, skill-building, etc.)\n   - Determine event duration (half-day, full-day, multi-day)\n   - Set overall budget\n   - Identify target outcomes/success metrics\n\n## 2. **Logistics & Scheduling**\n   - Choose date(s) that work for majority of attendees\n   - Book venue (size, capacity, accessibility, amenities)\n   - Arrange transportation if needed\n   - Secure parking or transit information\n   - Plan catering/meals\n\n## 3. **Activity Planning**\n   - Research team building activities suitable for 50 people\n   - Select 3-5 activities that align with objectives\n   - Coordinate with activity vendors/facilitators\n   - Create activity schedule with time allocations\n   - Prepare materials and equipment needed\n\n## 4. **Budget Management**\n   - Get quotes from venues and vendors\n   - Allocate funds across categories (venue, food, activ

In [90]:
# Saves the generated results in the results.json file

with open('results.json', 'w') as f:
    json.dump(results, f, indent=2)

## Lets Build The Grader

There are basically 3 types of graders:
- Model Grader
- Code Grader
- Human Grader

In [81]:
# Lets Build Our First Model Grader where the model grades the prompt

def grade_by_model(test_case, output):
    # Now we will implement an eval prompt
    eval_prompt = f"""
    You are an expert task deconstructor. Evaluate these AI generated subtasks for a particular task.

    Task: {test_case['task']}
    Solution: {output}

    Provide evaluation as a structured Json object with:
    - "strengths": an array of 1-3 key strengths
    - "weaknesses": an array of 1-3 key areas of improvement
    - "reasoning": a concise explanation of your assesment 
    - "score": a number between 1-10
    """
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")

    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [82]:
# After this we need to update our run_test_case function to call the grader
# so now we will update it above 

In [83]:
# Now that we have updated the run_test_case function above we will finally calculate an average score across all test cases: